In [1]:
platformID = 'FBE'


In [2]:
from datetime import datetime
import pandas as pd

import os 

## import helper 

In [3]:
import sys
from pathlib import Path

try:
    # Works in Python scripts
    helper_path = Path(__file__).resolve().parent.parent / "helper"
except NameError:
    # Works in Jupyter notebooks
    helper_path = Path().resolve().parent / "helper"

sys.path.insert(0, str(helper_path))

# Now import your modules
from functions import execute_sql_query, compare_or_update_reference
import test_functions

from config import gam_info

In [12]:
# country
country_cols = ['YT-_FBE_codes', 'PlaceID']
country_codes = pd.read_excel(f"../../{gam_info['lookup_file']}", sheet_name='CountryID',
                             keep_default_na=False)[country_cols]
# week 
week_cols = ['w/c']
week_tester = pd.read_excel(f"../../{gam_info['lookup_file']}", sheet_name='GAM Period')
week_tester['w/c'] = pd.to_datetime(week_tester['w/c'])

today = pd.Timestamp.today().normalize()
last_monday = today - pd.Timedelta(days=(today.weekday() % 7))
week_tester = week_tester[week_tester['w/c'] < last_monday]

# social media accoutns
channel_cols=['Channel ID']
dtype_dict = {'Channel ID': 'str',
              'Linked FB Account': 'str'}
socialmedia_accounts = pd.read_excel(f"../../{gam_info['lookup_file']}", dtype=dtype_dict,
                                     sheet_name='Social Media Accounts new')

#socialmedia_accounts = socialmedia_accounts[socialmedia_accounts['Year'] == gam_info['file_timeinfo']]
socialmedia_accounts = socialmedia_accounts[socialmedia_accounts['PlatformID'] == platformID]
socialmedia_accounts = socialmedia_accounts[socialmedia_accounts['Status'] == 'active']
channel_ids = socialmedia_accounts['Channel ID'].unique().tolist()

### RUN TESTS
test_functions.test_lookup_files(country_codes, country_cols, [f"{platformID}_1c_0", f"{platformID}_1c_1", f"{platformID}_1c_2"], 
                                 test_step="lookup files - ensuring country codes is correct")

test_functions.test_lookup_files(week_tester, ['w/c'], [f"{platformID}_1c_3", f"{platformID}_1c_4", f"{platformID}_1c_5"], 
                                 test_step = "lookup files - ensuring week tester is correct")

test_functions.test_lookup_files(socialmedia_accounts, ['Channel ID'], [f"{platformID}_1c_6", f"{platformID}_1c_7", f"{platformID}_1c_8"], 
                                 test_step = "lookup files - ensuring social media accounts is correct")


✅ Test FBE_1c_0 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test FBE_1c_1 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test FBE_1c_2 passed: No missing values in lookup.
...updating logbook...

✅ Test FBE_1c_3 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test FBE_1c_4 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test FBE_1c_5 passed: No missing values in lookup.
...updating logbook...

✅ Test FBE_1c_6 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test FBE_1c_7 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test FBE_1c_8 passed: No missing values in lookup.
...updating logbook...



In [13]:
week_tester

,YearGAE,WeekNumber_finYear,WeekNumber_calYear,Year,w/c,week_ending
0,2026,1,14,2025,2025-03-31,2025-04-06
1,2026,2,15,2025,2025-04-07,2025-04-13
2,2026,3,16,2025,2025-04-14,2025-04-20
3,2026,4,17,2025,2025-04-21,2025-04-27
4,2026,5,18,2025,2025-04-28,2025-05-04
5,2026,6,19,2025,2025-05-05,2025-05-11
6,2026,7,20,2025,2025-05-12,2025-05-18
7,2026,8,21,2025,2025-05-19,2025-05-25
8,2026,9,22,2025,2025-05-26,2025-06-01
9,2026,10,23,2025,2025-06-02,2025-06-08


## country

In [5]:

sql_query = f"""
    SELECT 
        week_commencing,
        page_id ,
        page_name,
        page_fans_country_total AS page_fans_country,
        country_code AS fb_metric_breakdown
    FROM
        redshiftdb.central_insights.adverity_social_facebook_page_fans_by_country
    WHERE
        week_commencing Between '{gam_info['w/c_start']}' and '{gam_info['w/c_end']}'
        ;
    """
file = f"../data/raw/{platformID}/{gam_info['file_timeinfo']}_{platformID}_country_redshift_extract.csv"

try: 
    df = execute_sql_query(sql_query)
    df['page_id'] = df['page_id'].astype(str)
    df.to_csv(file, index=False, na_rep='')
except:
    print("no redshift connection using last time queried")

facebook_country_raw = pd.read_csv(file, keep_default_na=False)

facebook_country_raw['page_id'] = facebook_country_raw['page_id'].astype(str)
facebook_country_raw['week_commencing'] = pd.to_datetime(facebook_country_raw['week_commencing'])
facebook_country_raw = facebook_country_raw.rename(columns={'page_id': 'Channel ID',
                                                            'page_name': 'Channel Name',
                                                            'week_commencing': 'w/c',
                                                            'fb_metric_breakdown': 'YT-_FBE_codes'})

### RUN TESTS
# missing page_ids
test_functions.test_filter_elements_returned(facebook_country_raw, 
                                             channel_ids, 
                                             'Channel ID', 
                                             f"{platformID}_1c_9",
                                             "Check that all page IDs are found in SQL")


# missing weeks per page_id
test_functions.test_weeks_presence_per_account(key='w/c',
                                               id_column='Channel ID',
                                               main_data=facebook_country_raw,
                                               week_lookup=week_tester[['w/c']],
                                               test_number=f"{platformID}_1c_10",
                                               test_step="Check all weeks present for each account")

# missing values per week / page id 
test_functions.test_non_null_and_positive(facebook_country_raw, 
                           numeric_columns=['page_fans_country'], 
                           test_number=f"{platformID}_1c_11",
                           test_step='Check no missing values in page fans column from redshift returned')

# test for duplicate entries 
test_functions.test_duplicates(facebook_country_raw, ['Channel ID', 'w/c'], 
                               test_number=f"{platformID}_1c_12",
                               test_step='Check no duplicates from redshift returned')


no redshift connection using last time queried
...testing Channel ID...
❌ Test FBE_1c_9 failed: not all elements from lookup were retrieved in dataset - decide if they are really missing or if these accounts are inactive 
...updating logbook...

❌ Test FBE_1c_10 failed: Missing completed weeks detected.
...updating logbook...

✅ Test FBE_1c_11 passed: No NaN and all numeric values > 0.
...updating logbook...

❌ Test FBE_1c_12 failed: The following combinations occur more than once a week
...updating logbook...



/Users/brunsd01/BBC Dropbox/Domi Bruns/BBC - InfoLab - Lumen 2020/digiGAM_ytd/digiGAM/helper/test_functions.py:66: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  week_lookup[key] = pd.to_datetime(week_lookup[key])


In [6]:
# filter to relevant pageID's
facebook_country = facebook_country_raw[facebook_country_raw['Channel ID'].isin(channel_ids)]

test_functions.test_inner_join(facebook_country, 
                               country_codes, 
                               ['YT-_FBE_codes'], 
                               f"{platformID}_1c_13",
                               test_step='calculating country %')

facebook_country = facebook_country.merge(country_codes[['YT-_FBE_codes', 'PlaceID']], 
                                          on='YT-_FBE_codes', 
                                          how='left').drop(columns=['YT-_FBE_codes'])

# Group by specified columns and sum the fb_metric_value
facebook_country_sum = facebook_country.groupby(['Channel ID', 'w/c'])\
                                        .agg(Sum_page_fans_country=('page_fans_country', 'sum'))\
                                        .reset_index()
facebook_country = facebook_country.merge(facebook_country_sum, 
                                          how='inner', on= ['Channel ID', 'w/c'])
test_functions.test_inner_join(facebook_country, facebook_country_sum, 
                               ['Channel ID', 'w/c'], 
                               f"{platformID}_1c_14", 
                               test_step='calculating country %')

facebook_country['country_%'] = facebook_country['page_fans_country']/facebook_country['Sum_page_fans_country']

### RUN TESTS
test_functions.test_percentage(facebook_country, 
                               ['Channel ID', 'w/c'], 
                               f"{platformID}_1c_15", 
                               test_step='summing country % per week & account')


Inner join test FBE_1c_13 failed: Issues found.
Issues with df_left (rows present in df_left but not in df_right)
Issues with df_right (rows present in df_right but not in df_left)
...updating logbook...

✅ Inner join test FBE_1c_14 successful: No issues found.
...updating logbook...

...updating logbook...



In [7]:
file_path = f"../data/processed/{platformID}"
os.makedirs(file_path, exist_ok=True)

cols = ['Channel ID', 'Channel Name', 'w/c', 'PlaceID', 'country_%']
#facebook_country[cols].to_csv(f"{file_path}/{gam_info['file_timeinfo']}_{platformID}_REDSHIFT_COUNTRY.csv",
#                        index=None)

In [8]:
#testing differences
stored_file = pd.read_csv(f"{file_path}/{gam_info['file_timeinfo']}_{platformID}_REDSHIFT_COUNTRY.csv")
stored_file['Channel ID'] = stored_file['Channel ID'].astype(str)
stored_file['w/c'] = pd.to_datetime(stored_file['w/c'])
combined = facebook_country[cols].merge(stored_file, on=['Channel ID', 'Channel Name', 'w/c', 'PlaceID'], how='outer', 
                                        suffixes=['_new', '_stored'] )
combined['diff'] = combined['country_%_new'] - combined['country_%_stored']
combined[(combined['diff'] > 0.01) | (combined['diff'] < -0.01)]

,Channel ID,Channel Name,w/c,PlaceID,country_%_new,country_%_stored,diff
